In [1]:
! pip install numpy torch h5py slmsuite tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 8.8 MB/s eta 0:00:00


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from slmsuite.holography.algorithms import Hologram

# =========================================================
# TARGET GENERATOR (GOOD QUALITY)
# =========================================================
def generate_target(N=256):
    x = np.linspace(-1, 1, N)
    X, Y = np.meshgrid(x, x)

    r = np.sqrt(X**2 + Y**2)

    # asymmetric pattern (IMPORTANT)
    ring = np.exp(-((r - 0.3)**2)/0.02)
    blob = np.exp(-((X+0.2)**2 + (Y-0.1)**2)/0.05)

    target = ring + 0.6 * blob

    target += 0.01*np.random.randn(N, N)
    target = np.clip(target, 0, None)

    target /= target.sum()

    return target


# =========================================================
# SOLVER
# =========================================================
def solve_hologram(target, iterations=500):

    hologram = Hologram(target)

    hologram.optimize(
        method="GS",   # BEST default
        maxiter=iterations,
        feedback="computational",
        stat_groups=["computational"],
        verbose=True
    )

    phase = hologram.get_phase()
    farfield = hologram.get_farfield()

    return phase, farfield


# =========================================================
# VISUALIZATION (CORRECT)
# =========================================================
def visualize(target, phase, farfield):

    pred_intensity = np.abs(farfield)**2
    pred_intensity /= pred_intensity.sum()

    error = np.abs(pred_intensity - target)

    plt.figure(figsize=(12,8))

    plt.subplot(2,2,1)
    plt.imshow(target)
    plt.title("Target")

    plt.subplot(2,2,2)
    plt.imshow(pred_intensity)
    plt.title("Reconstructed")

    plt.subplot(2,2,3)
    plt.imshow(phase)
    plt.title("Phase (SLM)")

    plt.subplot(2,2,4)
    plt.imshow(error)
    plt.title("Error")

    plt.tight_layout()
    plt.show()


# =========================================================
# DATASET GENERATOR
# =========================================================
def generate_dataset(num_samples=10):

    dataset = []

    for i in tqdm(range(num_samples)):
        target = generate_target()

        phase, farfield = solve_hologram(target, iterations=200)

        dataset.append({
            "target": target,
            "phase": phase
        })

        visualize(target, phase, farfield)

    return dataset


# =========================================================
# RUN
# =========================================================
if __name__ == "__main__":
    dataset = generate_dataset(5)

Output hidden; open in https://colab.research.google.com to view.